# Phoenix4All Star \&amp; Planetary/Stellar Catalogue

TauREx |version| ships with two previously-external plugins integrated directly into the codebase:

1. **Phoenix4All** — a modern, versatile interface to PHOENIX stellar atmosphere models that supports multiple back-end sources (SVO, STScI Synphot, Göttingen HiResFITS) and replaces the legacy ``PhoenixStar``.
2. **taurex-catalogue** — automatic loading of planet and star parameters from a local CSV file or from the ExoMAST API, via the mixin pattern.

This notebook explores both features and shows how they simplify realistic forward-model setups.

## Setup and Imports

We start by importing the necessary modules.  The ``_shared`` helper (used by other notebooks in this series) sets up the opacity cache with cross-section and CIA data from ExoMol and HITRAN.

In [ ]:
import sys
from pathlib import Path

# Locate the _shared helper and ensure the source tree is importable.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "taurex").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Append examples/notebooks to pick up _shared.py
NOTEBOOKS_DIR = PROJECT_ROOT / "examples" / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from _shared import build_base_components, ensure_opacity_data
from taurex.constants import RSOL

# Ensure opacity files are available (download=False uses cached files)
ensure_opacity_data(download=False)

## 1.  Comparing Stellar Spectra

We start by comparing the black-body spectrum used by the default ``BlackbodyStar`` with a realistic PHOENIX spectrum obtained via ``Phoenix4AllStar``.

The ``Phoenix4AllStar`` class supports several data sources:

* ``"svo"`` — Spanish Virtual Observatory (default), with ``model_name="bt-settl-cifist"``.
* ``"synphot"`` — STScI Synphot.
* ``"hiresfits"`` — Göttingen HiResFITS (the legacy ``PhoenixStar`` data).

Each source downloads model files on demand and caches them via Astropy's download cache, so the first call may be slow but subsequent calls are fast.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Wavenumber grid covering the near-IR to MIR (0.3-30 um)
wngrid = np.linspace(300, 30000, 10000)

# --- BlackbodyStar ---
from taurex.stellar import BlackbodyStar
bb_star = BlackbodyStar(temperature=5778, radius=1.0)
bb_star.initialize(wngrid)

# --- Phoenix4AllStar (SVO source) ---
from taurex.stellar import Phoenix4AllStar
p4a_star = Phoenix4AllStar(temperature=5778, radius=1.0, metallicity=0.0,
                           source="svo", model_name="bt-settl-cifist")
p4a_star.initialize(wngrid)

wlgrid = 10000.0 / wngrid[::-1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(wlgrid, bb_star.sed[::-1], lw=1.5, label="Blackbody (5778 K)")
ax1.plot(wlgrid, p4a_star.sed[::-1], lw=1.5, label="Phoenix4All (SVO)")
ax1.set_xscale("log")
ax1.set_xlabel("Wavelength (µm)")
ax1.set_ylabel("SED (W m⁻² sr⁻¹ cm)")
ax1.set_title("Stellar SED")
ax1.legend()
ax1.grid(alpha=0.2)

ax2.plot(wlgrid, (p4a_star.sed[::-1] / bb_star.sed[::-1] - 1) * 100, lw=1.5, color="C1")
ax2.set_xscale("log")
ax2.set_xlabel("Wavelength (µm)")
ax2.set_ylabel("Deviation (%)")
ax2.set_title("Phoenix4All vs. Blackbody")
ax2.axhline(0, color="gray", ls="--", lw=0.5)
ax2.grid(alpha=0.2)

plt.tight_layout()

### 1.1  Transmission model with Phoenix4AllStar

Now we plug the ``Phoenix4AllStar`` into a full forward model — the same ``TransmissionModel`` workflow shown in the [transmission basics notebook](https://escience-taurex.github.io/taurex3/examples/03_transmission_basics.html) — and compute a spectrum.

In [ ]:
# Build the usual atmospheric components (isothermal T, simple P, chemistry ...)
from taurex.chemistry import ConstantGas, TaurexChemistry
from taurex.model import TransmissionModel
from taurex.contributions import AbsorptionContribution
from taurex.pressure import SimplePressureProfile
from taurex.temperature import Isothermal
from taurex.planet import Planet

# Atmosphere
iso_t = Isothermal(T=1500.0)
press = SimplePressureProfile(nlayers=60, atm_min_pressure=1e-5, atm_max_pressure=1e5)
chem = TaurexChemistry(fill_gases=["H2", "He"], ratio=0.1756)
chem.addGas(ConstantGas(molecule_name="H2O", mix_ratio=1e-3))
chem.addGas(ConstantGas(molecule_name="CH4", mix_ratio=1e-4))
press.compute_pressure_profile()
iso_t.initialize_profile(planet=None, nlayers=press.nLayers, pressure_profile=press.profile)

# Planet
planet = Planet(planet_mass=0.74, planet_radius=1.38)

# Star -- Phoenix4All
star = Phoenix4AllStar(temperature=6117, radius=1.16, metallicity=0.0,
                       source="svo", model_name="bt-settl-cifist")

chem.set_star_planet(star=star, planet=planet)
chem.initialize_chemistry(nlayers=press.nLayers,
                          temperature_profile=iso_t.profile,
                          pressure_profile=press.profile)

# Forward model
tm = TransmissionModel(planet=planet, temperature_profile=iso_t,
                       chemistry=chem, star=star, pressure_profile=press)
tm.add_contribution(AbsorptionContribution())
tm.build()

wn, rprs, tau, _ = tm.model()
wl = 10000 / wn[::-1]

plt.figure(figsize=(7, 4))
plt.plot(wl, rprs[::-1], lw=1.5)
plt.xscale("log")
plt.xlabel("Wavelength (µm)")
plt.ylabel("$(R_p/R_s)^2$")
plt.title("Transmission spectrum (Phoenix4All star)")
plt.grid(alpha=0.2)

## 2.  Using the Catalogue Module

The ``taurex.catalogue`` sub-package provides **mixin classes** that automatically load planet and star parameters from a data source.  There are two families:

* ``*CatalogueFile`` — reads from a local CSV/TSV catalogue.
* ``*CatalogueExomast`` — fetches from the `ExoMAST API <https://exo.mast.stsci.edu>`_.

These are combined with the base ``Planet`` or ``BlackbodyStar`` (or ``Phoenix4AllStar``) classes via the ``enhance_class`` mixin pattern, producing a new class that inherits from both.

### 2.1  Loading from ExoMAST

The ``PlanetCatalogueExomast`` and ``StarCatalogueExomast`` mixins query the ExoMAST API by planet name and populate the parameters automatically.

In [ ]:
from taurex.mixin import enhance_class
from taurex.planet import Planet
from taurex.stellar import BlackbodyStar
from taurex.catalogue import PlanetCatalogueExomast, StarCatalogueExomast

# Create enhanced planet and star classes
PlanetE = enhance_class(Planet, PlanetCatalogueExomast,
                        planet_name="WASP-121 b")
StarE = enhance_class(BlackbodyStar, StarCatalogueExomast,
                      planet_name="WASP-121 b")

print(f"Planet mass:   {PlanetE.mass:.3f} M_jup")
print(f"Planet radius: {PlanetE.radius:.3f} R_jup")
print(f"Planet sma:    {PlanetE.distance:.3f} AU")
print(f"Star mass:     {StarE.mass / 1.989e30:.3f} M_sun")
print(f"Star radius:   {StarE.radius / 6.957e8:.3f} R_sun")
print(f"Star Teff:     {StarE.temperature:.0f} K")
print(f"Star [Fe/H]:   {StarE.metallicity:.2f}")
print(f"Star distance: {StarE.distance:.1f} pc")

### 2.2  Loading from a local CSV file

If you have a curated catalogue — for example the Edwards et al. 2020 compilation — you can place it as a CSV file and use the ``PlanetCatalogueFile`` and ``StarCatalogueFile`` mixins.  The CSV should follow the naming convention described in the [TauREx catalogue documentation](https://escience-taurex.github.io/taurex3/user/support_data_formats.html).

We'll create a small sample catalogue inline for this demonstration.

In [ ]:
import tempfile, csv

# --- Create a tiny sample catalogue ---
sample_csv = """Planet Name,Planet Mass [MJ],Planet Radius [RJ],Planet Orbital Period [days],Planet Semi-major Axis [AU],Planet Eccentricity,Planet Inclination [deg],Star Name,Star Radius [RS],Star Distance [pc],Star Metallicity,Star Mass [MS],Star Effective Temperature [K]
TRAPPIST-1 d,0.00031,0.0772,4.049610,0.02227,0.00837,89.89,TRAPPIST-1,0.117,12.474,0.023,0.089,2511
"""

# Also keep the CSV comma-separated
sample_csv_path = "/tmp/sample_catalogue.csv"
with open(sample_csv_path, "w") as f:
    f.write(sample_csv.replace(",", ","))

from taurex.catalogue import PlanetCatalogueFile, StarCatalogueFile

PlanetF = enhance_class(Planet, PlanetCatalogueFile,
                         catalogue_file=sample_csv_path)
StarF = enhance_class(BlackbodyStar, StarCatalogueFile,
                       catalogue_file=sample_csv_path)

print("--- Planet (from file) ---")
for attr in ("mass", "radius", "distance", "temperature"):
    val = getattr(PlanetF, attr, None)
    if val is not None:
        print(f"  {attr} = {val:.4g}")

print("\n--- Star (from file) ---")
for attr in ("mass", "radius", "temperature", "metallicity", "distance"):
    val = getattr(StarF, attr, None)
    if val is not None:
        # Star mass/radius in SI; convert to solar units for display
        if attr == "mass":
            print(f"  mass = {val / 1.989e30:.4g} M_sun")
        elif attr == "radius":
            print(f"  radius = {val / 6.957e8:.4g} R_sun")
        else:
            print(f"  {attr} = {val:.4g}")

### 2.3  Forward model with catalogue-loaded parameters

Finally, we combine the catalogue-loaded planet and star into a full transmission forward model — demonstrating a completely parameter-file-free workflow powered by ExoMAST.

In [ ]:
# Reload from ExoMAST for a well-studied hot Jupiter
PlanetW = enhance_class(Planet, PlanetCatalogueExomast,
                        planet_name="WASP-121 b")
StarW = enhance_class(BlackbodyStar, StarCatalogueExomast,
                      planet_name="WASP-121 b")

# Atmosphere
iso_t2 = Isothermal(T=1800.0)
press2 = SimplePressureProfile(nlayers=60, atm_min_pressure=1e-5, atm_max_pressure=1e5)
chem2 = TaurexChemistry(fill_gases=["H2", "He"], ratio=0.1756)
chem2.addGas(ConstantGas(molecule_name="H2O", mix_ratio=1e-3))
chem2.addGas(ConstantGas(molecule_name="CO2", mix_ratio=1e-5))
press2.compute_pressure_profile()
iso_t2.initialize_profile(planet=None, nlayers=press2.nLayers, pressure_profile=press2.profile)
chem2.set_star_planet(star=StarW, planet=PlanetW)
chem2.initialize_chemistry(nlayers=press2.nLayers,
                           temperature_profile=iso_t2.profile,
                           pressure_profile=press2.profile)

tm2 = TransmissionModel(planet=PlanetW, temperature_profile=iso_t2,
                        chemistry=chem2, star=StarW, pressure_profile=press2)
tm2.add_contribution(AbsorptionContribution())
tm2.build()
wn2, rprs2, tau2, _ = tm2.model()
wl2 = 10000 / wn2[::-1]

plt.figure(figsize=(7, 4))
plt.plot(wl2, rprs2[::-1], lw=1.5, color="C2")
plt.xscale("log")
plt.xlabel("Wavelength (µm)")
plt.ylabel("$(R_p/R_s)^2$")
plt.title("Transmission spectrum — WASP-121 b (catalogue-loaded)")
plt.grid(alpha=0.2)
None

## Summary

In this notebook we have seen:

* **Phoenix4AllStar** — the replacement for the old ``PhoenixStar`` that supports multiple data sources (SVO, Synphot, HiResFITS) with a consistent API.
* **Catalogue mixins** — classes that auto-populate planet/star parameters from:
  * `ExoMAST API` — by planet name (``PlanetCatalogueExomast``, ``StarCatalogueExomast``).
  * `Local CSV file` — using the standard TauREx naming convention (``PlanetCatalogueFile``, ``StarCatalogueFile``).
* **The mixin pattern** — ``enhance_class(Base, Mixin, **kwargs)`` produces a new class combining both.

These features together let you write *parameter-file-free* forward model scripts: planetary/stellar data are fetched on the fly, and stellar spectra are computed with realistic PHOENIX models.

**Note:** The first time you run a ``Phoenix4AllStar`` with a given source/model combination, it downloads the spectral grid (a few hundred MB).  Subsequent runs load from the local Astropy cache.  The ExoMAST queries are lightweight and only need internet access on the first call per session.